# 57 Auxiliary MediaPipe nasion route

Compares the automatic MediaPipe nasion detector to the manually picked nasion across the seven thesis subjects. Feeds Table 4.8 (confidence + distance) and Figure 4.7 (midline Y-profile plot for Subject 17) of the thesis.

**BRANCH REQUIRED.** The auto-nasion code was moved out of `main` in commit 95a7a4c and now lives on the `auto-detection-pipeline` branch. Before running this notebook:

```

cd /home/ma7/BA/cedalion/cedalion

git checkout auto-detection-pipeline

```

Then run this notebook. Remember to switch back to `main` afterwards so the other Results notebooks keep working.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))
from _thesis_helpers import (
    SUBJECTS, subject_paths, load_raw, load_anon, load_landmarks,
    available_subjects, missing_report, run_pipeline,
)

import numpy as np
import pandas as pd
import xarray as xr

OUT_DIR = pathlib.Path('thesis_results_out')
OUT_DIR.mkdir(exist_ok=True)

import matplotlib.pyplot as plt

## 1. Import the auto-nasion detector

This import only succeeds on the `auto-detection-pipeline` branch. On `main` it raises `ImportError` and the notebook cleanly reports that the branch switch is required.

In [2]:
try:

    from cedalion.geometry.photogrammetry.anonymization.nasion_detector import (

        detect_nasion_auto,

    )

    HAS_AUTO = True

except ImportError as err:

    HAS_AUTO = False

    print('Auto-nasion detector is not on this branch:')

    print(f'  {err}')

    print('Run: git checkout auto-detection-pipeline')

Auto-nasion detector is not on this branch:
  No module named 'cedalion.geometry.photogrammetry.anonymization.nasion_detector'
Run: git checkout auto-detection-pipeline


## 2. Per-subject comparison

In [3]:
rows = []

profiles = {}  # subject -> midline Y-profile for plotting

if HAS_AUTO:

    for n in SUBJECTS:

        paths = subject_paths(n)

        if not (paths.raw_exists and paths.landmarks_exist):

            print(f'skipping Subject{n}: missing raw or landmarks')

            continue

        surface_raw = load_raw(n)

        landmarks_raw = load_landmarks(n)



        Nz_manual = landmarks_raw.sel(label='Nz').pint.dequantify().values



        result = detect_nasion_auto(surface_raw)

        Nz_auto = result.position

        conf = float(result.confidence)

        dist = float(np.linalg.norm(Nz_manual - Nz_auto))

        outcome = 'success' if conf >= 0.3 else 'fallback'



        rows.append({

            'subject': n,

            'confidence': conf,

            'distance_to_manual_mm': dist,

            'outcome': outcome,

        })

        # Keep the midline profile if the detector exposes it

        profiles[n] = getattr(result, 'profile', None)

        print(rows[-1])

## 3. Summary table

In [4]:
df = pd.DataFrame(rows).sort_values('subject').reset_index(drop=True) if rows else pd.DataFrame()

df

""


## 4. Figure 4.7: midline Y-profile for Subject 17

Plots the midline Y(z) profile with the auto-detected and manually picked nasion annotated. The figure is scoped to Subject 17 only because thesis data-sharing rules restrict subject-identifiable figures to Subject 17, even though the numeric table (Table 4.8) covers all seven subjects.

In [5]:
FIG_SUBJECT = 17

if FIG_SUBJECT in profiles and profiles[FIG_SUBJECT] is not None:

    prof = profiles[FIG_SUBJECT]

    r = df[df.subject == FIG_SUBJECT].iloc[0]

    fig, ax = plt.subplots(figsize=(8, 4))

    ax.plot(prof['z'], prof['y'], label='midline Y(z)')

    ax.set_title(f'Subject{FIG_SUBJECT} (conf={r.confidence:.2f}, d={r.distance_to_manual_mm:.1f} mm)')

    ax.set_xlabel('Z (mm)')

    ax.set_ylabel('Y (mm)')

    ax.legend()

    fig.tight_layout()

    out = OUT_DIR / 'auxiliary_nasion_profile.pdf'

    fig.savefig(out, dpi=200, bbox_inches='tight')

    print(f'Wrote {out}')

else:

    print(f'No profile for Subject{FIG_SUBJECT}; skipping figure.')

No profile for Subject17; skipping figure.


## 5. Save CSV

In [6]:
if len(df):

    out = OUT_DIR / 'auxiliary_nasion.csv'

    df.to_csv(out, index=False)

    print(f'Wrote {out}')